# データ準備
## 前提
- プロジェクトルートで uv sync（取得はネット必須）
‐ raw / external は一度書いたら上書きしない（再取得は別ファイル名）
- Statcast / MLB Stats API はスプリット名のみ（例: `2025_regular.parquet`）。再取得日は notebook 実行日またはコミットで記録
- pybaseball は pandas を返す → 保存・分析は polars に載せ替える

In [7]:
# 共通セットアップ
import json
from datetime import date

import polars as pl
import statsapi
from pybaseball import chadwick_register, playerid_lookup, statcast

from analysis_project.paths import data_dir, ensure_parent_dir

FETCH_DATE = date.today().strftime("%Y%m%d")  # 例: 20260921


# 山本由伸の ID 固定（マイルストーン1）
## Keys
- key_mlbam=808967
- key_fangraphs=33825

In [11]:
# 山本由伸の ID をそれぞれ取得する
ids_yoshi_yamamoto = playerid_lookup("yamamoto", "yoshinobu")
print(ids_yoshi_yamamoto)


# 山本由伸の ID 固定（マイルストーン2）
## 例（2026-03 時点の lookup 結果）:
## key_mlbam=808967, key_fangraphs=33825

YAMAMOTO_MLBAM = 808967
YAMAMOTO_FANG = 33825

# ID データ取得
## 保存するディレクトリを作成
register_dir = data_dir() / "external" / "register"

## 山本由伸の ID 一覧を parquet で保存する
output_path = ensure_parent_dir(register_dir / "yamamoto_ids.parquet")
pl.from_pandas(ids_yoshi_yamamoto).write_parquet(output_path)

# 全選手の ID Register を取得し、保存する（初回のみ　結合用）
reg_all_players_path = ensure_parent_dir(register_dir / "chadwick_register.parquet")

if not reg_all_players_path.exists(): # 初回のみ実行される
    reg_all_players = chadwick_register()
    pl.from_pandas(reg_all_players).write_parquet(reg_all_players_path)


  name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
0  yamamoto  yoshinobu     808967  yamay001  yamamyo01          33825   

   mlb_played_first  mlb_played_last  
0            2024.0           2026.0  
Gathering player lookup table. This may take a moment.


# データ取得
- Statcast: 投球データ
- MLB Stats API: シーズン投手成績（`/stats`）・試合記録

In [12]:


# 投球データの取得関数
def fetch_statcast(start_dt:str, end_dt:str, label:str) -> pl.DataFrame:
    """label 例："2025_regular", "2025_post"

    Args:
        start_dt (str): _description_
        end_dt (str): _description_
        label (str): _description_

    Returns:
        pl.DataFrame: _description_
    """
    path = ensure_parent_dir(
        data_dir() / "external" / "statcast" / f"{label}.parquet"
    )

    if path.exists():
        return pl.read_parquet(path)

    # 投球データを取得する
    ## 投球データを statcast で取得する pdf -> pandas+dataframe の意味
    pdf = statcast(start_dt=start_dt, end_dt=end_dt) # 全投球・1球1行
    df = pl.from_pandas(pdf)
    df.write_parquet(path)
    return df

# 試合記録・API JSON の保存
def save_json(obj, name: str) -> None:
    """JSON ファイルを保存する"""
    path = ensure_parent_dir(
        data_dir() / "raw" / "mlb_statsapi" / f"{name}.json"
    )
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


# MLB Stats API — シーズン投手成績（リーグ全体。limit 省略時は API 側で 50 件のみ）
def fetch_mlb_pitching_stats(
    season: int,
    game_type: str,
    split_label: str,
    player_pool: str = "QUALIFIED",
    limit: int = 1000,
) -> dict:
    """
    split_label 例: regular / post
    game_type は R（レギュラー）または P（ポスト）。
    """
    pool_suffix = "_all" if player_pool.upper() == "ALL" else ""
    json_name = f"pitching_{season}_{split_label}{pool_suffix}"
    path = data_dir() / "raw" / "mlb_statsapi" / f"{json_name}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    payload = statsapi.get(
        "stats",
        {
            "stats": "season",
            "group": "pitching",
            "season": season,
            "gameType": game_type,
            "playerPool": player_pool,
            "limit": limit,
        },
    )
    save_json(payload, json_name)
    return payload

# 試合ごと boxscore（gamePk は schedule から）
# box = statsapi.boxscore(game_pk)
# save_json(box, f"boxscore_{game_pk}")


In [10]:
# データの取得
fetch_statcast("2025-03-18", "2025-09-28", "2025_regular")
fetch_statcast("2025-09-30", "2025-11-01", "2025_post")

# 投手成績（順位母集団: QUALIFIED / 全登板: ALL）
fetch_mlb_pitching_stats(2025, "R", "regular", player_pool="QUALIFIED")
fetch_mlb_pitching_stats(2025, "R", "regular", player_pool="ALL")
fetch_mlb_pitching_stats(2025, "P", "post", player_pool="ALL")

# 例： 2025 年ドジャースの試合一覧（WS 特定は game_type / 日付で絞る）
games = statsapi.schedule(season=2025, team=119)
save_json(games, "schedule_2025_LAD")

# DuckDBの作成